In [1]:
import numpy as np
import torch as tn

In [30]:
shape = [8, 4, 4, 5, 3, 10]
#shape = [2, 5, 7, 5, 10, 2]
mean = tn.zeros(shape)
std = tn.ones(shape)
tensor = tn.normal(mean, std)
#tensor = 2 * tn.ones(shape)

In [31]:
# X       @ A       = B
# (m x n) @ (n x p) = (m x p)
m = 600
n = 5
p = 32

A = tn.normal(tn.zeros([n, p]), tn.ones([n, p]))
B = tn.normal(tn.zeros([m, p]), tn.ones([m, p]))

X = tn.linalg.lstsq(A.t(), B.t()).solution.t()
print(X.shape)

#dist = tn.dist(X @ A, B)
#dist = tn.dist(A.t() @ X.t(), B.t())
dist = tn.norm(B - X @ A) 
print(f'dist = {dist}')

torch.Size([600, 5])
dist = 125.49178314208984


In [32]:
def full_tensor_from_tr(tr):
    # compute product of all cores sequentially
    tensor = tr[0]
    for factor in tr[1:-1]:
        tensor = tn.tensordot(tensor, factor, dims=([-1], [0]))
    # END for
    tensor = tn.tensordot(tensor, tr[-1], dims=([0, -1], [-1, 0]))

    return tensor
# END full_tensor_from_tr()

In [33]:
def compute_subchain(cores, k, d):
    right_subchain = None
    left_subchain = None
    
    # compute right subchain
    right_start = k + 2
    if right_start <= d - 1:
        right_subchain = cores[right_start]
        for j in range(right_start + 1, d):
            right_subchain = tn.tensordot(right_subchain, cores[j], dims=([-1], [0]))
        # END for
    # END if

    # compute left subchain
    left_end = k - 1
    if left_end >= 0:
        left_start = 1 if (k + 1) % d == 0 else 0
        left_subchain = cores[left_start]
        for j in range(left_start + 1, left_end + 1):
            left_subchain = tn.tensordot(left_subchain, cores[j], dims=([-1], [0]))
        # END for
    # END if

    # combine left and right subchains
    if right_subchain != None and left_subchain != None:
        subchain = tn.tensordot(right_subchain, left_subchain, dims=([-1], [0]))
    else:
        subchain = right_subchain if right_subchain != None else left_subchain
    # END if

    # combine all interor indices
    # note that this is the same as
    # subchain.reshape([subchain.shape[0], np.prod(subchain.shape[1:-1]) , subchain.shape[-1]])
    subchain = subchain.flatten(1, -2)    
    
    return subchain
# END compute_subchain()


def mode_k_unfolding(tensor, k):
    dims = list(range(tensor.dim()))
    permute_dims = [dims[k]] + dims[k + 1:] + dims[:k]
    
    return tensor.permute(permute_dims).flatten(1)
# END mode_k_unfolding()


def rank_chop(s, delta):
    # this is the rank selection procedure
    # used by torchtt -- zhao 2016 didn't
    # specifiy the procedure they used
    
    if tn.linalg.norm(s) == 0:
        return 1
    # END if
    
    if delta < 0:
        return s.size(dim=0)
    # END if

    rank = s.size(dim=0) - 1

    cumsum = np.cumsum(np.abs(s.numpy()[::-1])**2)[::-1]
    rank = np.argmax(cumsum < delta ** 2)

    rank = rank if rank > 0 else 1
    rank = s.size(dim=0) if cumsum[-1] > delta ** 2 else rank

    return rank
# END rank_chop()

In [57]:
def tr_bals(tensor, eps=1e-10, max_iter=100):
    ### step 0
    ### get meta info
    d = tensor.dim()
    dims = list(range(d))
    shape = tensor.shape
    norm_tensor = tn.norm(tensor).item()
    
    ### step 1
    ### init ranks
    ranks = tn.ones(d + 1, dtype=int)

    ### step 2
    ### init cores
    cores = []
    for k in range(0, d):
        core_shape = [ranks[k], shape[k], ranks[k + 1]]
        mean = tn.zeros(core_shape)
        std = tn.ones(core_shape)
        
        core = tn.normal(mean, std)
        cores.append(core)
    # END for

    ### step 3
    ### loop until convergance or max_iters reached
    err = float('inf')
    iterations = 0
    while err >= eps and iterations < max_iter:
        k = iterations % d
        print(f'k = {k}, iter = {iterations}')

        ### step 4
        ### get ZZneq = ZZ^{\neq (k,k+1)}
        ZZneq = compute_subchain(cores, k, d)

        ### step 5
        ### get Zneq = Z_{[2]}^{\neq (k,k+1)}
        ### i.e. perform mode-2 unfolding on ZZneq
        Zneq = mode_k_unfolding(ZZneq, 1)

        ### step 6
        ### calculate solution to least squares problem
        ### Z = arg min ||T - Z @ Zneq^t||
        ### a least sqaures problem posed this way can be
        ### solved via tn.linalg.lstsq(Zneq, T.t())
        ### here, T = T_[k,k+1], the mode-[k,k+1]
        ### unfolding of the original tensor, and
        ### Z = Z_{(2)}^{(k,k+1)}
        
        # first, we get the mode-[k,k+1] unfolding of
        # the original tensor -- this step is not exactly
        # as it is in Zhao 2016 because of a probably typo
        # this is my best guess how it is supposed to work
        permute_dims = [dims[k], dims[(k + 1) % d]]
        #permute_dims = [dims[(k + 1) % d], dims[k]]
        if k + 1 == d:
            permute_dims += dims[1:k]
        else:
            permute_dims += dims[k + 2:] + dims[:k]
        # END if
        T = tensor.permute(permute_dims).flatten(2).flatten(0, 1)

        # now, we solve the least squares problem and 
        # take the transpose of the solution to get Z
        Z = tn.linalg.lstsq(Zneq, T.t()).solution.t()

        # if this step was done correctly, we should get
        # that T \approx Z @ Zneq.t() as we iterate
        #dist = tn.dist(Z.t(), tn.linalg.pinv(Zneq) @ T.t())
        #dist = tn.dist(Zneq @ Z.t(), T.t())
        #dist = tn.dist(T, Z @ Zneq.t()) / tn.norm(T)
        #norm = tn.norm(T - Z @ Zneq.t())# / tn.norm(T)
        #print(T.shape, Z.shape, Zneq.t().shape)
        #print(f'error from lstsq = {dist}')
        

        ### step 7
        ### obtain ZZ = ZZ^{(k,k+1)} by folding Z
        ### ZZ will have shape [r_k, n_k, n_{k+1}, r_{k+2}]
        #ZZ = Z.reshape([ranks[k], 
        #                shape[k], 
        #                shape[(k + 1) % d], 
        #                ranks[(k + 2) % d]])
        ZZ = Z.reshape([shape[k], 
                        shape[(k + 1) % d],
                        ranks[k],
                        ranks[(k + 2) % d]]).permute([2, 0, 1, 3])

        ### step 8 
        ### obtain Ztilde = \tilde{Z}^{(k,k+1)} by reshaping
        ### ZZ to have shape [r_k n_k, n_{k+1} r_{k+2}]
        Ztilde = ZZ.flatten(0, 1).flatten(1)

        ### step 9
        ### approximate Ztilde via delta-truncated SVD
        
        # delta depends on the current error, calculate
        # that first
        #reconstructed = full_tensor_from_tr(cores)
        #err = (tn.norm(tensor - reconstructed) / norm_tensor).item()
        #print(f'{err}\n{ranks}\n')

        # calculate delta
        delta = max(eps * norm_tensor / np.sqrt(d),  # should be err, not eps
                    eps * norm_tensor / np.sqrt(d)).item()
        if k == d-1:
            delta = np.sqrt(2) * delta

        # calculate SVD and truncate
        U, S, Vt = tn.linalg.svd(Ztilde, full_matrices=True)
        print(S)
        rank_delta = rank_chop(S, delta)
        U = U[:, :rank_delta]
        S = S[:rank_delta]
        Vt = Vt[:rank_delta, :]
        print(S)

        ### step 10
        ### update shared rank and ensure that
        ### first/last ranks are equal
        ranks[k + 1] = rank_delta
        if k + 1 == d:
            ranks[0] = ranks[-1]
        # END if

        ### step 11
        ### reshape and assign left core
        #cores[k] = U.reshape([ranks[k], 
        #                      shape[k], 
        #                      ranks[k + 1]])
        cores[k] = U.reshape([shape[k], 
                              ranks[k], 
                              ranks[k + 1]]).permute([1, 0, 2])

        ### step 12
        ### reshape and assign right core
        #cores[(k + 1) % d] = (tn.diag(S) @ Vt).reshape([ranks[k + 1], 
        #                                                shape[(k + 1) % d], 
        #                                                ranks[(k + 2) % d]])
        #cores[(k + 1) % d] = (tn.diag(S) @ Vt).reshape([shape[(k + 1) % d],
        #                                                ranks[k + 1], 
        #                                                ranks[(k + 2) % d]]).permute([1, 0, 2])
        cores[(k + 1) % d] = (tn.diag(S) @ Vt).reshape([ranks[k + 1], 
                                                        ranks[(k + 2) % d],
                                                        shape[(k + 1) % d]]).permute([0, 2, 1])

        iterations += 1

        # probbaly want to calc error here anyway?
        reconstructed = full_tensor_from_tr(cores)
        err = (tn.norm(tensor - reconstructed) / norm_tensor).item()
        print(f'{err}\n{ranks}\n')
    # END while

    if iterations == max_iter:
        print(f'Max iterations reached. Current err = {err}')
    # END if
    
    return cores, ranks
# END tr_bals()


cores, ranks = tr_bals(tensor, eps=1e-3, max_iter=20)
print(ranks)

k = 0, iter = 0
tensor([0.2352, 0.1933, 0.1434, 0.0851])
tensor([0.2352, 0.1933, 0.1434, 0.0851])
0.9994024634361267
tensor([1, 4, 1, 1, 1, 1, 1])

k = 1, iter = 1
tensor([0.7242, 0.5630, 0.4203, 0.3568])
tensor([0.7242, 0.5630, 0.4203, 0.3568])
1.000697374343872
tensor([1, 4, 4, 1, 1, 1, 1])

k = 2, iter = 2
tensor([1.6223, 1.0695, 0.9536, 0.7257, 0.4599])
tensor([1.6223, 1.0695, 0.9536, 0.7257, 0.4599])
1.0007224082946777
tensor([1, 4, 4, 5, 1, 1, 1])

k = 3, iter = 3
tensor([2.0936, 1.6995, 1.0332])
tensor([2.0936, 1.6995, 1.0332])
1.0006797313690186
tensor([1, 4, 4, 5, 3, 1, 1])

k = 4, iter = 4
tensor([6.1518, 4.0543, 3.7400, 3.6613, 2.5154, 2.1207, 1.8126, 1.1998, 0.6949])
tensor([6.1518, 4.0543, 3.7400, 3.6613, 2.5154, 2.1207, 1.8126, 1.1998, 0.6949])
0.9998677968978882
tensor([1, 4, 4, 5, 3, 9, 1])

k = 5, iter = 5
tensor([37.8369, 33.8157, 32.6452, 32.3324, 30.7312, 29.0285, 27.5974, 26.8773,
        25.4629, 25.3020, 24.3068, 23.6796, 22.6710, 21.7626, 21.2743, 19.7603,
     

In [58]:
full = full_tensor_from_tr(cores)
print(full.shape)

err = tn.norm(tensor - full) / (tn.norm(tensor) + 1e-32)
print(err.item())

torch.Size([8, 4, 4, 5, 3, 10])
1.0174864530563354


In [59]:
full[0, 0, 0, 0, 0, 0].item()

-0.0770750641822815

In [51]:
tensor[0, 0, 0, 0, 0, 0].item()

1.806091547012329